# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I will choose **Structured Content Archetype Clustering**.

Why this one? Because a content inventory of any real size has hundreds of pages, and nobody can eyeball which ones behave alike. Right now the only tools are gut feel and single-metric sorts ("top 10 by traffic").

This lane turns scattered pageviews, engagement, and freshness signals into a small set of repeatable types — champions, hidden gems, stale pages, and so on — using only safe structured metrics. That's honest work: the release has no article text, so I can cluster on numbers and be upfront that it's not "semantic" clustering, just behavioral clustering from safe metrics.

## 2. The question: decision, action, cost of a wrong call


*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Using the four-question frame:

**What decision does this improve?** Which pages a content team should act on next, and how — protect, improve, rewrite, merge, prune, or monitor. Not "predict decline," but "where should limited editor time go this sprint."

**Who acts on it, and what do they do?** A content strategist or SEO editor triaging a backlog. Given a cluster label and its profile (typical traffic, engagement, freshness), they decide whether a page gets a refresh, gets merged into a stronger page, gets left alone, or gets pruned.

**What does a wrong answer cost?**

* False "prune" call → deleting a hidden gem (low traffic, high engagement) that was quietly working — lost equity, wasted rebuild cost later.
* False "protect" call → leaving a stale visible page alone because it looks like a champion on one metric, while it's actually declining — lost traffic that compounds.
* Miscalling a cannibalization-risk cluster as "improve" → editor invests hours rewriting a page that's competing with itself, not underperforming.

  So the cost isn't abstract — it's editor hours misallocated and slow-bleed traffic losses that go unnoticed because nothing dramatic triggered a review.

**Why does clustering (not a plain rule) help?** A single threshold rule ("traffic < X → prune") ignores interaction effects — a low-traffic page can be a hidden gem if engagement is high, or dead weight if it isn't. The pattern is real (pages do fall into a handful of recognizable behavior types) but too tangled across multiple axes (traffic, engagement, freshness, token count) to hand-write as if-statements. Clustering finds that structure; it doesn't predict anything, and I'll be careful not to claim it does.

**One-paragraph frame:**
> For content strategists deciding which pages to prioritize this sprint, we will build cluster profiles and archetype labels from structured warehouse metrics (traffic, engagement, freshness, token counts — no article text), scoring no explicit target since this is unsupervised, evaluated by silhouette score plus human sense-check of whether cluster profiles map sensibly to known page examples. A wrong call costs misallocated editor hours or a quietly-declining page left unattended. A plain rule isn't enough because the archetypes emerge from combinations of signals, not single thresholds. We will claim only observed, descriptive, decision-support groupings — never semantic meaning, never causal claims, never predictions of future performance.



## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [18]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows / pages:", len(df))
print(df.dtypes)
print("Unique clients:", df["client_id"].nunique())

Rows / pages: 30000
content_id                 object
client_id                  object
search_volume             float64
competition               float64
competition_level          object
cpc                       float64
content_type               object
main_intent                object
word_count                float64
char_count                float64
provider_used              object
model_used                 object
impressions_90d             int64
clicks_90d                  int64
pageviews_90d               int64
sessions_90d                int64
users_90d                   int64
engaged_sessions_90d        int64
ai_sessions_90d             int64
scroll_events_90d           int64
days_with_impressions       int64
days_with_sessions          int64
impressions_last_30d        int64
clicks_last_30d             int64
sessions_last_30d           int64
impressions_prev_30d        int64
clicks_prev_30d             int64
sessions_prev_30d           int64
content_age_days            

In [19]:
# --- Skew: does a small slice of pages dominate traffic? ---
top10_share = (
    df.sort_values("sessions_90d", ascending=False)
      .head(int(len(df) * 0.10))["sessions_90d"].sum()
    / df["sessions_90d"].sum()
)
print(f"Top 10% of pages by sessions_90d account for {top10_share:.1%} of total sessions")

# --- Hidden-gem hint: low traffic, but real engagement ---
low_traffic = df["sessions_90d"] <= df["sessions_90d"].quantile(0.25)
hidden_gem_like = low_traffic & (df["engagement_rate"] >= df["engagement_rate"].median())
print(f"Share of pages that are low-traffic but above-median engagement: {hidden_gem_like.mean():.1%}")

# --- Stale-visible hint: getting impressions, but trending down ---
stale_visible_like = (df["impressions_last_30d"] > 0) & (df["trend_direction"] == "down")
print(f"Share of pages with impressions but a declining trend: {stale_visible_like.mean():.1%}")

# --- Weak/no-demand hint: essentially invisible in search ---
threshold = df["impressions_90d"].quantile(0.1)
weak_demand = df["impressions_90d"] <= threshold
print(f"Share of pages with weak impressions in 90d: {weak_demand.mean():.1%}")

Top 10% of pages by sessions_90d account for 65.7% of total sessions
Share of pages that are low-traffic but above-median engagement: 26.9%
Share of pages with impressions but a declining trend: 49.6%
Share of pages with weak impressions in 90d: 10.2%


In [20]:
print(df["impressions_90d"].describe())
print(df["trend_direction"].value_counts())

count     30000.000000
mean       5200.366300
std       16838.019547
min           1.000000
25%          81.000000
50%         731.000000
75%        3615.250000
max      517715.000000
Name: impressions_90d, dtype: float64
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


the 4 real numbers to lead with are
* **65.7%** of sessions concentrated in the top 10% of pages
* **26.9%** hidden-gem candidates (low traffic, above-median engagement)
* **49.6%** declining-trend pages (54% of all pages are trending `down`)
* **10.2%** weak-impression pages (bottom decile, by construction)

those are enough on their own to make the case that clustering (not averages, not a single threshold) is worth the next 7 weeks.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**Can say:**

* These are observed behavioral groupings from structured metrics on this date range.
* Cluster profiles are descriptive — "cluster 3 has median sessions of X and near-zero engagement."
* Archetype names are a lens, applied after inspecting real cluster contents, not before.
* Findings are decision-support: they suggest where to look next, not what will happen.

**Can never say:**

* That this is semantic clustering — there's no article text in the release, only metrics, buckets, and token counts. Calling it semantic would misrepresent the method.
* Anything causal — clustering doesn't tell you why a page performs the way it does, only that it resembles other pages that do.
* Anything predictive of future performance, like "this page will decline" — no target was observed over a future window, so no forecast is licensed.
* That a cluster label is a fixed, true identity for a page — it's a snapshot lens on this data pull, and pages can shift clusters.
* That this work is not by any mean a way to predict Google's Algorithm nor try to claim it.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.